# 02 — Data Preparation

**Objective**: turn a raw transactions table (user_id, item_id, [rating]) into the sparse `interactions` matrix `AryColBringModelTrainer.fit()` expects, and validate it before training.

**Audience**: anyone plugging their own transaction data into AryColBring for the first time.

> `norm_exchange()` (the real encoding function, `src/models/arycolbring/assist/bloatdata.py`) uses DuckDB internally for the ID-encoding query so it scales to tens of millions of rows. Section 2 below reproduces the exact same *transformation* in plain pandas so this notebook is runnable without a DuckDB install; Section 3 shows the real call for reference.

## 1. A raw transactions table

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp

np.random.seed(0)

# Stand-in for a raw transactions export -- same shape norm_exchange() expects:
# arbitrary (non-contiguous) user_id / item_id values, one row per interaction.
raw_user_ids = np.random.choice([f"U-{n}" for n in range(1, 41)], size=500)
raw_item_ids = np.random.choice([f"SKU-{n}" for n in range(100, 130)], size=500)
transactions = pd.DataFrame({"user_id": raw_user_ids, "item_id": raw_item_ids,
                             "qty": np.random.randint(1, 5, size=500)})
# Missing-value handling: drop rows missing either id (norm_exchange requires both).
transactions = transactions.dropna(subset=["user_id", "item_id"]).drop_duplicates()
print(transactions.shape)
transactions.head()

## 2. Encoding to contiguous integer ids + building the sparse matrix

This is exactly what `norm_exchange()` does under the hood (its DuckDB query performs the same `DENSE_RANK()`-style encoding), reproduced here in pandas:

In [ ]:
def encode_interactions(data: pd.DataFrame, user_col="user_id",
                        item_col="item_id", rating_col=None):
    """Pure-pandas stand-in for norm_exchange(). Returns
    (interactions: coo_matrix, user_ids: np.ndarray, item_ids: np.ndarray) --
    same return shape as the real function."""
    user_codes, user_ids = pd.factorize(data[user_col], sort=True)
    item_codes, item_ids = pd.factorize(data[item_col], sort=True)
    values = data[rating_col].to_numpy(dtype="float32") if rating_col else \
             np.ones(len(data), dtype="float32")
    interactions = sp.coo_matrix((values, (user_codes, item_codes)),
                                 shape=(len(user_ids), len(item_ids)))
    return interactions, user_ids.to_numpy(), item_ids.to_numpy()


interactions, user_ids, item_ids = encode_interactions(transactions)
print("interactions:", interactions.shape, "nnz =", interactions.nnz)
print("user_ids[:5] =", user_ids[:5])
print("item_ids[:5] =", item_ids[:5])

## 3. Validating the matrix before training

The real validation/description utilities (`src/models/arycolbring/assist/`) are pure Python except for one DuckDB query inside `describe_interactions()`. `validate_sparse_matrix()` has no DuckDB dependency at all and is safe to call directly:

```python
from src.models.arycolbring.assist import validate_sparse_matrix, describe_interactions

validate_sparse_matrix(interactions.tocsr(), min_interactions_per_user=1)
stats = describe_interactions(interactions.tocsr()).iloc[0]
print(stats[["n_users", "n_items", "nnz", "density"]])
```

A minimal, dependency-free version of the same shape/finite-value checks, runnable here:

In [ ]:
def basic_matrix_checks(mat: sp.spmatrix) -> dict:
    csr = mat.tocsr()
    n_users, n_items = csr.shape
    nnz = csr.nnz
    return {
        "n_users": n_users,
        "n_items": n_items,
        "nnz": nnz,
        "density": nnz / (n_users * n_items) if n_users and n_items else 0.0,
        "has_nan": bool(np.isnan(csr.data).any()) if nnz else False,
        "min_interactions_per_user": int(np.asarray(csr.sum(axis=1)).min()) if n_users else 0,
    }


checks = basic_matrix_checks(interactions)
for k, v in checks.items():
    print(f"{k:>26}: {v}")
assert not checks["has_nan"], "interactions matrix must not contain NaN values"
print("\nAll basic checks passed.")

## 4. Train / test split

`AryColBringModelTrainer.fit()` takes a `test_ratio` and handles the split internally (see `src/models/arycolbring/trainer.py`), holding out a fraction of *interactions* (not users/items) at random. A simple illustration of the same idea:

In [ ]:
def train_test_split_interactions(mat: sp.coo_matrix, test_ratio: float = 0.2, seed: int = 0):
    rng = np.random.default_rng(seed)
    n = mat.nnz
    test_mask = rng.random(n) < test_ratio
    train = sp.coo_matrix((mat.data[~test_mask], (mat.row[~test_mask], mat.col[~test_mask])),
                          shape=mat.shape)
    test  = sp.coo_matrix((mat.data[test_mask], (mat.row[test_mask], mat.col[test_mask])),
                          shape=mat.shape)
    return train.tocsr(), test.tocsr()


train_mat, test_mat = train_test_split_interactions(interactions, test_ratio=0.25)
print(f"train nnz = {train_mat.nnz}, test nnz = {test_mat.nnz}, "
      f"ratio = {test_mat.nnz / interactions.nnz:.2f}")

## Summary

- Raw (user_id, item_id[, rating]) rows -> `encode_interactions()` -> contiguous-index sparse matrix + id-mapping arrays (the real `norm_exchange()` does the same, via a DuckDB query).
- Validate with `validate_sparse_matrix()` / `describe_interactions()` before training -- catches empty rows/columns and NaNs early.
- Split with `AryColBringModelTrainer.fit(interactions, test_ratio=...)`, or manually as shown above.

**Next**: `03_Training_AryColBring.ipynb` for the full training + evaluation loop using this prepared matrix.